AI4I 2020 Predictive Maintenance
Multi-Class Failure Classification Pipeline

Classes:
 - 0 - No Failure
 - 1 - Tool Wear Failure (TWF)
 - 2 - Heat Dissipation Failure (HDF)
 - 3 - Power Failure (PWF)
 - 4 - Overstrain Failure (OSF)
 - 5 - Random Failure (RNF)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

import warnings
warnings.filterwarnings("ignore")

: 

In [ ]:
# 1. LOAD DATA

DATA_PATH = "ai4i2020.csv" 
df = pd.read_csv(DATA_PATH)

print("\nDataset loaded successfully.")
print("Shape:", df.shape)

In [ ]:
# 2. CREATE MULTI-CLASS TARGET

def create_multiclass_label(row):
    if row["TWF"] == 1:
        return 1
    if row["HDF"] == 1:
        return 2
    if row["PWF"] == 1:
        return 3
    if row["OSF"] == 1:
        return 4
    if row["RNF"] == 1:
        return 5
    return 0  # No failure

df["failure_class"] = df.apply(create_multiclass_label, axis=1)

print("\nFailure class distribution:")
print(df["failure_class"].value_counts().sort_index())

In [ ]:
# 3. DROP UNUSED COLUMNS

drop_cols = [
    "UDI",
    "Product ID",
    "Machine failure",
    "TWF", "HDF", "PWF", "OSF", "RNF"
]

df = df.drop(columns=drop_cols)

In [ ]:
# 4. FEATURE / TARGET SPLIT

X = df.drop(columns=["failure_class"])
y = df["failure_class"]

categorical_features = ["Type"]
numerical_features = [col for col in X.columns if col not in categorical_features]

In [ ]:
# 5. PREPROCESSING PIPELINE

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(drop="first"), categorical_features)
    ]
)

In [ ]:
# 6. TRAIN / TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("\nTrain/Test split completed.")

In [ ]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train, y_train = sm.fit_resample(X_train, y_train)

# Verify the new class distribution
import pandas as pd
print(pd.Series(y_train).value_counts())

In [ ]:
# 7. MODELS TO EVALUATE

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, multi_class="ovr"),
    "Support Vector Machine": SVC(kernel="rbf"),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=7),
    "Decision Tree": DecisionTreeClassifier(max_depth=10),
    "Random Forest": RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    class_weight='balanced',
    random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        random_state=42
    )
}

In [ ]:
# 8. TRAIN & EVALUATE MODELS

results = {}

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training: {name}")
    print(f"{'='*50}")

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    results[name] = acc

    print(f"Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

In [ ]:
# 9. BEST MODEL SUMMARY

print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)

for model, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{model:<25} : {acc:.4f}")

best_model = max(results, key=results.get)
print("\n🏆 BEST MODEL:", best_model)
print("🎯 BEST ACCURACY:", results[best_model])

In [ ]:
# 10. CONFUSION MATRIX FOR BEST MODEL

print("\nGenerating confusion matrix for best model...")

best_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", models[best_model])
])

best_pipeline.fit(X_train, y_train)
y_best_pred = best_pipeline.predict(X_test)

cm = confusion_matrix(y_test, y_best_pred)
print("\nConfusion Matrix:")
print(cm)

print("\nPipeline execution completed successfully.")


In [ ]:
import joblib

joblib.dump(best_pipeline, "model.pkl")
print("Model saved as model.pkl")